# Bouquet + TokaMaker: DIII-D Equilibrium Reconstruction and Perturbation

This notebook demonstrates the full **bouquet** workflow for producing
families of perturbed tokamak equilibria from EFIT-format input files:

1. **Load inputs** — read g-files (magnetic equilibria) and p-files
   (kinetic profiles) using the bouquet I/O modules.
2. **Reconstruct equilibria** — reproduce each g-file equilibrium in
   TokaMaker via an inverse solve, decomposing the toroidal current
   density into inductive ($j_{\text{ind}}$) and bootstrap
   ($j_{\text{BS}}$) components.
3. **Verify reconstructions** — compare TokaMaker’s solution against
   the original g-file using `plot_tokamaker_comparison()`.
4. **Generate perturbed families** — use Gaussian-process-regression
   (GPR) sampling to produce statistically consistent variations of
   the kinetic and current-density profiles, each iterated to match
   $I_p$ and $l_i$.
5. **Visualize results** — inspect the perturbed families with
   `plot_bouquet()`.

### Prerequisites

| Package | Purpose |
|---|---|
| [OpenFUSIONToolkit / TokaMaker](https://github.com/OpenFUSIONToolkit) | Grad–Shafranov solver |
| [h5py](https://www.h5py.org/) | HDF5 equilibrium database |
| bouquet | This package — I/O, reconstruction, perturbation, plotting |

### Input files (in this directory)

| File | Description |
|---|---|
| `TkMkr_D3Dlike_Hmode_BSamp=*.geqdsk` | TokaMaker-generated DIII-D–like H-mode equilibria |
| `TkMkr_D3Dlike_Hmode_BSamp=*.peqdsk` | Corresponding kinetic profiles in Osborne p-file format |
| `DIIID_mesh.h5` | Pre-built TokaMaker finite-element mesh for DIII-D |

## 1. Imports

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = "retina"

# --- TokaMaker ---
tokamaker_python_path = '/Users/danielburgess/Desktop/plasma/OpenFUSIONToolkit/build_release'
if tokamaker_python_path is not None:
    sys.path.append(os.path.join(tokamaker_python_path, 'python'))

from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.TokaMaker import TokaMaker
from OpenFUSIONToolkit.TokaMaker.meshing import load_gs_mesh
from OpenFUSIONToolkit.TokaMaker.util import create_power_flux_fun

# --- Bouquet ---
from bouquet import (
    read_geqdsk,
    reconstruct_equilibrium,
    generate_bouquet,
    initialize_equilibrium_database,
    store_equilibrium,
    calc_cylindrical_li_proxy,
    plot_tokamaker_comparison,
    plot_bouquet, plot_traces,
)
from bouquet.io.pfile import read_pfile
from bouquet import new_uncertainty_profiles
from bouquet import sigmoid_length_scale

## 2. User configuration

All tunable parameters are collected here so that the rest of the
notebook can be run without modification.  Adjust these to your use
case:

- **`geqdsks` / `pfiles`**: matched lists of g-file and p-file paths.
- **`frac_*`**: fractional 1-$\sigma$ uncertainties for GPR perturbation.
- **`n_equils`**: number of perturbed equilibria per baseline.
- **`*_ls`**: GPR correlation length scales (0–1 in $\psi_N$).

In [ ]:
# ----- Input file lists (must be same length and in matching order) -----
geqdsks = [
    'TkMkr_D3Dlike_Hmode_BSamp=1.0.geqdsk',
    'TkMkr_D3Dlike_Hmode_BSamp=1.25.geqdsk',
]
pfiles = [
    'TkMkr_D3Dlike_Hmode_BSamp=1.0.peqdsk',
    'TkMkr_D3Dlike_Hmode_BSamp=1.25.peqdsk',
]

header = 'TkMkr_D3Dlike_Hmode'   # HDF5 database name (without .h5)

# ----- Uncertainty envelope parameters for GPR perturbation -----
# new_uncertainty_profiles(psi_N, frac, falloff_exp) generates a
# radially-varying fractional envelope:
#
#   sigma(psi_N) = (frac * (1 - psi_N)^falloff_exp + shelf) * profile(psi_N)
#
# Larger falloff_exp => tighter constraint near the pedestal/edge
# (where Thomson scattering is better constrained) and wider in the core.
frac_ne   = 0.20;  falloff_ne   = 0.5   # electron density
frac_te   = 0.20;  falloff_te   = 0.5   # electron temperature
frac_ni   = 0.30;  falloff_ni   = 0.3    # ion density
frac_ti   = 0.30;  falloff_ti   = 0.3    # ion temperature
frac_jphi = 0.15;                        # toroidal current density, using a flat envelope instead
shelf = 0.025                            # minimum (edge) uncertainty floor (fraction of profile value) 

# ----- GPR correlation length scales (in psi_N units) -----
n_ls = 0.5    # density
t_ls = 0.4    # temperature
j_ls = 0.25   # current density

# ----- Perturbation settings -----
n_equils = 15       # perturbed equilibria per baseline
jBS_scale_range = [0.8, 1.2]  # bootstrap current multiplicative range (uniform draw)
pad_psi  = 1e-3     # LCFS psi padding for TokaMaker queries

# ----- Ion species for p-file (Carbon impurity + D main + D beam) -----
ion_N = [6, 1, 1]
ion_Z = [6, 1, 1]
ion_A = [12, 2, 2]

## 3. Initialize TokaMaker

We create a single `TokaMaker` instance for the session, load the
pre-built DIII-D mesh, define conductor and coil regions, and call
`setup()` with the toroidal field $F_0 = R_0 B_0$ read from the first
g-file.

> **Note:** Only one `TokaMaker` instance can exist per Python kernel.

In [ ]:
myOFT = OFT_env(nthreads=1)   # keep at 1: bit-reproducible, no oversubscription
mygs  = TokaMaker(myOFT)

# Load mesh
mesh_pts, mesh_lc, mesh_reg, coil_dict, cond_dict = load_gs_mesh('DIIID_mesh.h5')
mygs.setup_mesh(mesh_pts, mesh_lc, mesh_reg)
mygs.setup_regions(cond_dict=cond_dict, coil_dict=coil_dict)

In [ ]:
# Read F0 = R0*B0 from the first g-file
eqdsk_ref = read_geqdsk(geqdsks[0])
F0 = abs(eqdsk_ref.R_center * eqdsk_ref.B_center)

mygs.setup(order=3, F0=F0)
mygs.settings.maxits = 200
mygs.settings.pm = False
mygs.update_settings()

# Vertical stability coil pair (F9A/F9B antisymmetric)
mygs.set_coil_vsc({'F9A': 1.0, 'F9B': -1.0})

## 4. Coil regularization

TokaMaker’s inverse solver adjusts PF coil currents to match the
isoflux and saddle-point targets.  We regularize the solution by
providing target coil currents from a prior TokaMaker solve of this
equilibrium.

> **Note:** These target currents are specific to this DIII-D mesh and
> equilibrium family.  For a different machine or mesh, you would need
> to determine appropriate coil currents (e.g. from an EFIT
> reconstruction or a prior TokaMaker run).

In [ ]:
# Target coil currents [MA-turns] for this DIII-D mesh
target_currents = {
    'ECOILA': -0.977888676757812 / 61.0,
    'ECOILB': -0.962711173828125 / 61.0,
    'F1A':  0.115971984375,      'F1B':  0.128368578125,
    'F2A':  0.05980789453125,    'F2B':  0.0763701328125,
    'F3A': -0.03076001171875,    'F3B': -0.02234583203125,
    'F4A': -0.0401077421875,     'F4B': -0.13314096875,
    'F5A':  0.000723009033203125,'F5B':  0.000399709045410156,
    'F6A': -0.1178582578125,     'F6B': -0.15356990625,
    'F7A':  0.0341264296875,     'F7B':  0.0415082109375,
    'F8A': -0.05660116015625,    'F8B': -0.05138975390625,
    'F9A':  0.236625375,         'F9B':  0.252380265625,
}

# Convert MA-turns -> A-turns and build regularization
regularization_terms = []
for name, val in target_currents.items():
    current_A = val * 1e6
    weight = 1.0  # uniform weighting on all coils
    regularization_terms.append(
        mygs.coil_reg_term({name: 1.0}, target=current_A, weight=weight)
    )

# Small weight on the virtual VSC to allow up-down adjustment
regularization_terms.append(
    mygs.coil_reg_term({'#VSC': 1.0}, target=0.0, weight=1e-2)
)

mygs.set_coil_reg(reg_terms=regularization_terms)

## 5. Reconstruct baseline equilibria

For each (g-file, p-file) pair we:

1. Load the magnetic equilibrium with `read_geqdsk()`.
2. Load kinetic profiles with `read_pfile()` and compute the impurity
   density (quasi-neutrality) and $Z_{\text{eff}}$.
3. Convert from p-file units (10$^{20}$/m$^3$, keV) to SI (m$^{-3}$,
   eV) as required by the reconstruction routine.
4. Call `reconstruct_equilibrium()` which:
   - Computes bootstrap current via TokaMaker’s Sauter model,
   - Fits a smooth inductive $j_\phi$ profile,
   - Iterates on the inductive scale factor until $l_i$ matches the
     g-file value,
   - Corrects residual $I_p$ drift.
5. Save the reconstructed equilibrium to the HDF5 database.

In [ ]:
# Initialize (or overwrite) the HDF5 database
db_path = f"{header}.h5"
if os.path.exists(db_path):
    os.remove(db_path)
    print(f"Deleted existing {db_path}")

db = initialize_equilibrium_database(header)

In [ ]:
from scipy.interpolate import interp1d

all_results = {}   # keyed by geqdsk filename

for idx, (geqdsk_file, pfile_file) in enumerate(zip(geqdsks, pfiles)):
    print(f"\n{'='*60}")
    print(f"Reconstructing [{idx}]: {geqdsk_file}")
    print(f"  Kinetic profiles: {pfile_file}")
    print(f"{'='*60}")

    # ---- 1. Load magnetic equilibrium ----
    eqdsk = read_geqdsk(geqdsk_file)

    # ---- 2. Load kinetic profiles from p-file ----
    pf = read_pfile(pfile_file)

    # Ensure ion species are set (needed for quasi-neutrality and Zeff)
    if pf.ion_species is None:
        pf.set_ion_species(N=ion_N, Z=ion_Z, A=ion_A)

    pf.compute_quasineutrality()
    psi_pf, Zeff = pf.compute_zeff()


    # ---- 3. Convert p-file units -> SI and interpolate onto g-file grid ----
    psi_N = eqdsk.psi_N  # g-file's psi_N grid (uniform, 0 to 1)

    ne_SI   = interp1d(psi_pf, pf.ne * 1e20,  fill_value='extrapolate')(psi_N)  # m^-3
    te_SI   = interp1d(psi_pf, pf.te * 1e3,   fill_value='extrapolate')(psi_N)  # eV
    ni_SI   = interp1d(psi_pf, pf.ni * 1e20,  fill_value='extrapolate')(psi_N)  # m^-3
    ti_SI   = interp1d(psi_pf, pf.ti * 1e3,   fill_value='extrapolate')(psi_N)  # eV
    Zeff_eq = interp1d(psi_pf, Zeff,           fill_value='extrapolate')(psi_N)
    Zeff_eq = np.clip(Zeff_eq, 1.0, None)  # Zeff >= 1 by definition

    # ---- 4. Set boundary shape targets ----
    isoflux_pts = np.column_stack([eqdsk.boundary_R, eqdsk.boundary_Z])
    isoflux_weights = np.ones(len(eqdsk.boundary_R)) * 500.0
    mygs.set_isoflux(isoflux_pts, weights=isoflux_weights)

    # ---- 5. Initial guess for inductive j_phi shape ----
    nr = len(psi_N)
    guess_jinductive = create_power_flux_fun(nr, 1.5, 1.5)['y']

    # ---- 6. Reconstruct ----
    result = reconstruct_equilibrium(
        mygs, eqdsk,
        ne_SI, te_SI, ni_SI, ti_SI, Zeff_eq,
        isoflux_pts, isoflux_weights, pad_psi,
        guess_jinductive=guess_jinductive,
        n_k=5,
        psi_bridge=0.99,
        rescale_j_BS=False,
        shelf_psi_N=0.0,
        initialize_psi=True,
    )
    all_results[geqdsk_file] = result

    # ---- 7. Save reconstructed equilibrium ----
    eqdsk_out = f"{header}_count={idx}.geqdsk"
    eqdsk_out_abs = os.path.abspath(eqdsk_out)
    mygs.save_eqdsk(eqdsk_out, nr=257, nz=257,
                    truncate_eq=False, lcfs_pad=pad_psi)

    li1 = mygs.get_stats(lcfs_pad=pad_psi, li_normalization='std')['l_i']
    li3 = mygs.get_stats(lcfs_pad=pad_psi, li_normalization='iter')['l_i']

    store_equilibrium(
        header, idx, eqdsk_out_abs,
        psi_N,
        result['j_phi_fit'],
        result['j_BS_used'],
        result['j_inductive_fit'],
        ne_SI, te_SI, ni_SI, ti_SI,
        np.zeros_like(ti_SI),  # w_ExB placeholder
        li1, li3,
    )

    # Clean up the temporary geqdsk (data is in HDF5 now)
    os.remove(eqdsk_out)
    print(f"  -> Stored as count={idx}, li(1)={li1:.4f}, li(3)={li3:.4f}")

### Check equilibrium $l_i$ values

Print the equilibrium $l_i$ (stored during reconstruction as
`result['li_final']`) and the cylindrical proxy $l_i$ for each
reconstructed baseline.  These are the targets that
`generate_bouquet()` will try to match.

> **Note:** Do not use `mygs.get_stats()` here — TokaMaker is
> stateful and only holds the last solved equilibrium, so querying
> it in a loop over multiple baselines would return the wrong $l_i$
> for all but the last one.

In [ ]:
from bouquet import calc_cylindrical_li_proxy

for idx, geqdsk_file in enumerate(geqdsks):
    result = all_results[geqdsk_file]
    psi_N = read_geqdsk(geqdsk_file).psi_N

    # Use the li stored during reconstruction, NOT mygs.get_stats()
    # which only reflects the last solved equilibrium.
    li_final = result['li_final']
    li_eqdsk = result['eqdsk_li']

    # Cylindrical proxy l_i (this is what the proxy filter compares)
    li_proxy = calc_cylindrical_li_proxy(mygs, result['j_phi_fit'], pad_psi)

    print(f"[{idx}] {geqdsk_file}")
    print(f"  l_i(1) [reconstructed]: {li_final:.6f}")
    print(f"  l_i(1) [g-file]:        {li_eqdsk.get('li(1)', 'N/A')}")
    print(f"  l_i (cyl proxy):        {li_proxy:.6f}")
    print(f"  proxy/reconstructed:    {li_proxy/li_final:.4f}  ({100*(li_proxy/li_final - 1):+.2f}%)")
    print()

## 6. Verify reconstructions

`plot_tokamaker_comparison()` overlays the TokaMaker solution against
the original g-file for each reconstructed equilibrium.  The comparison
panels include:

- Total $j_\phi$ with inductive and bootstrap components
- Pressure profile
- Kinetic profiles ($n_e$, $T_e$, $n_i$, $T_i$)
- Boundary shape deviation
- $l_i$ and $I_p$ error

Call with `plot_idx=None` for a multi-panel overview of all
reconstructions, or with a specific index for a single-shot deep dive.

In [ ]:
# Overview of all reconstructions
plot_tokamaker_comparison(mygs, all_results)

In [ ]:
# Detailed single-shot comparison for the first equilibrium
plot_tokamaker_comparison(mygs, all_results, plot_idx=0)

## 7. Generate perturbed equilibrium families

For each reconstructed baseline, `generate_bouquet()`:

1. Draws GPR-perturbed density, temperature, and $j_\phi$ profiles
   using the specified uncertainties and correlation lengths.
2. Recomputes the bootstrap current for each perturbed profile set
   (when `recalculate_j_BS=True`).
3. Iteratively adjusts the inductive current profile to match the
   baseline $l_i$ within the specified tolerance.
4. Archives each equilibrium (g-file bytes + profiles) to the HDF5
   database.

The `jBS_scale_range` parameter applies a random multiplicative factor
to the bootstrap current for each sample, drawn uniformly from the
specified range (e.g. [0.8, 1.2] gives ±20% variation).  This
captures uncertainty in the bootstrap model itself, independent of the
kinetic profile perturbations.

The resulting database is self-contained: it stores baseline profiles,
uncertainties, and all perturbed equilibria, so `plot_bouquet()` only
needs the file path.

> **Note:** For finer control of the $j_\phi$ correlation structure,
> bouquet also provides `sigmoid_length_scale()` which creates a
> spatially-varying GPR length scale — broad in the core and short
> near the edge to preserve narrow features like the bootstrap spike.
> Pass the result as `j_ls` instead of a scalar.  See
> `bouquet.sigmoid_length_scale` for details.

In [ ]:
for idx, geqdsk_file in enumerate(geqdsks):
    print(f"\n{'='*60}")
    print(f"Generating {n_equils} perturbed equilibria for [{idx}]: {geqdsk_file}")
    print(f"{'='*60}")

    eqdsk  = read_geqdsk(geqdsk_file)
    result = all_results[geqdsk_file]
    psi_N  = eqdsk.psi_N

    # Re-extract SI profiles from the reconstruction result
    ne_SI = result['ne']
    te_SI = result['te']
    ni_SI = result['ni']
    ti_SI = result['ti']
    Zeff_eq = result['Zeff']

    # Build radially-varying 1-sigma uncertainty profiles
    # Tighter in the pedestal where Thomson is well-constrained,
    # wider in the core and near the separatrix.
    sigma_ne   = new_uncertainty_profiles(psi_N, frac_ne,   falloff_exp=falloff_ne,   shelf=shelf) * ne_SI
    sigma_te   = new_uncertainty_profiles(psi_N, frac_te,   falloff_exp=falloff_te,   shelf=shelf) * te_SI
    sigma_ni   = new_uncertainty_profiles(psi_N, frac_ni,   falloff_exp=falloff_ni,   shelf=shelf) * ni_SI
    sigma_ti   = new_uncertainty_profiles(psi_N, frac_ti,   falloff_exp=falloff_ti,   shelf=shelf) * ti_SI
    sigma_jphi = frac_jphi * np.abs(result['j_phi_fit'])

    # Targets from the reconstruction
    # NOTE: Do NOT use mygs.get_stats() here — TokaMaker only holds
    # the last solved equilibrium.  Use the li stored during
    # reconstruction instead.
    Ip_target  = abs(eqdsk.Ip)
    l_i_target = result['li_final']
    print(f"  l_i target = {l_i_target:.6f} (from reconstruction)")

    # Read the baseline g-file and p-file as raw bytes for HDF5 archival
    with open(geqdsk_file, 'rb') as fh:
        baseline_eqdsk_raw = fh.read()
    pfile_file = pfiles[idx]
    with open(pfile_file, 'rb') as fh:
        pfile_raw_bytes = fh.read()

    # Restore isoflux targets for this equilibrium
    mygs.set_isoflux(result['isoflux_pts'], weights=result['weights'])

    # Generate the family
    diagnostics = generate_bouquet(
        mygs, psi_N, n_equils, header,
        result['j_phi_fit'],
        ne_SI, te_SI, ni_SI, ti_SI,
        sigma_ne, sigma_te, sigma_ni, sigma_ti, sigma_jphi,
        n_ls, t_ls, j_ls,
        Ip_target, l_i_target, Zeff_eq,
        input_jinductive=result['j_inductive_fit'],
        l_i_tolerance=10,
        l_i_proxy_threshold=12.5,
        psi_pad=pad_psi,
        constrain_sawteeth=True,
        recalculate_j_BS=True,
        jBS_scale_range=jBS_scale_range,
        pfile_bytes=pfile_raw_bytes,
        baseline_eqdsk_bytes=baseline_eqdsk_raw,
        baseline_pfile_bytes=pfile_raw_bytes,
        diagnostic_plots=False,
        scan_val=idx,
    )

    print(f"  -> {len(diagnostics)} equilibria archived")

## 8. Visualize perturbed families

`plot_bouquet()` reads the HDF5 database and overlays all perturbed
equilibria against the baseline.  Available modes:

| Mode | Panels |
|---|---|
| `"kinetic"` | $n_e$, $n_i$, $T_e$, $T_i$ with uncertainty bands |
| `"pressure"` | Thermal pressure |
| `"j-phi"` | Total $j_\phi$, $j_{\text{ind}}$, $j_{\text{BS}}$ |
| `"all"` | All of the above |

In [ ]:
# Plot all panels for the first baseline (scan_value=0)
plot_bouquet(header, scan_value=0, mode="all")

In [ ]:
# Plot all panels for the second baseline (scan_value=1)
plot_bouquet(header, scan_value=1, mode="all")

In [ ]:
# Trace plots: l_i, Ip, and boundary deviation across all equilibria
figs = plot_traces(header, scan_value='all')
plt.show()

## 9. Inspect individual equilibria and profiles

`plot_geqdsk_bouquet()` provides a quick overview of any g-file
(flux surfaces, pressure, q, j_phi, P' and FF').  `plot_pfile_bouquet()`
shows all kinetic profiles from a p-file.  Both support switching
between $\psi_N$ and $\rho$ coordinates via the `x_coord` flag.

In [ ]:
from bouquet import plot_geqdsk_bouquet

# Inspect both baseline g-files
plot_geqdsk_bouquet(geqdsks,x_coord="rho")

In [ ]:
# Inspect all perturbed g-files

plot_geqdsk_bouquet(h5path=f"{header}.h5", x_coord="rho")

In [ ]:
from bouquet import plot_pfile_bouquet

# Inspect the first baseline p-file (with rho coordinate)
eq_ref = read_geqdsk(geqdsks[0])
plot_pfile_bouquet(pfiles[0], x_coord="rho", eq=eq_ref)

In [ ]:
# Inspect all perturbed p-files

plot_pfile_bouquet(h5path=f"{header}.h5", scan_val=0, x_coord="psi_N")

## 10. Coil current variation

`plot_coil_currents()` shows how the PF coil currents vary across
the bouquet of perturbed equilibria.  This is useful for checking
that the shape control system is not being pushed to its limits.

In [ ]:
from bouquet import plot_coil_currents

# Coil currents for the first scan value
plot_coil_currents(header, scan_val=0)